# debugging_ci

The 95% bootstrap confidence intervals of some learning curves look very noisy,
which suggests the rapid changes in the mean are statistically significant.
Possible reasons:

1. The CI computation has a bug. The intended method is a bootstrap CI, with
   [rl-evaluation](https://github.com/rlai-lab/rl-evaluation/tree/main/rlevaluation)
   as the reference implementation.
2. The agent's or environment's randomness is not controlled properly, so many or
   all seeds covary, which could explain the variance in the CI.
3. The sample is noisy and changes rapidly over time. The CI computation does not
   take the temporal correlation of samples into account, and there is no bug.

The two components below had the noisiest bands, with their runs copied from the
parent experiments:

- `agent0_seaquest`: Agent0 on Seaquest from `benchmarking_atari_50m_realv1`.
- `dqn_asterix`: DQN on Asterix from `minatar_deterministic`.

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

from analysis.curves import return_curve, subsample
from analysis.stats import Interval, mean_ci, median_ti
from main import load_results

from config import EXPERIMENT

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots"

In [ ]:
# Load each component's stored runs.
AGENT0_SEAQUEST = "agent0_seaquest"
DQN_ASTERIX = "dqn_asterix"
results = {
    name: load_results(EXPERIMENT, name)
    for name in (AGENT0_SEAQUEST, DQN_ASTERIX)
}

In [ ]:
# Compute each component's per-seed return over steps, and its mean with a 95%
# bootstrap CI, over the whole run and nested slices of time. The first slice is
# SLICE_FRACTION of the run centered at SLICE_POSITION of it, and each next one is
# the middle SLICE_FRACTION of the previous. Each span is subsampled on its own,
# so it keeps detail the wider spans drop.
SLICES = 4
SLICE_FRACTION = 0.1
SLICE_POSITION = 0.75


def centered(span, fraction, position=0.5):
    start, end = span
    center = start + (end - start) * position
    half_width = (end - start) * fraction / 2
    return round(center - half_width), round(center + half_width)


def summarize(curve, span):
    start, end = span
    steps, returns = subsample(curve[:, start:end])
    return steps + start, returns, mean_ci(returns)


spans = {}
curves = {}
for name, runs in results.items():
    total = EXPERIMENT.component(name).config.AGENT_HYPERS.TOTAL_TIMESTEPS
    spans[name] = [(0, total), centered((0, total), SLICE_FRACTION, SLICE_POSITION)]
    for _ in range(SLICES - 1):
        spans[name].append(centered(spans[name][-1], SLICE_FRACTION))
    curve = return_curve(runs.reward, runs.done)
    curves[name] = [summarize(curve, span) for span in spans[name]]

In [ ]:
# Style for the learning-curve plots, as in minatar_deterministic. Every piece
# of text shares one size.
FONT_SIZE = 32
YLABEL_SIZE = 40
STYLE = {
    "font.family": "Times New Roman",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": FONT_SIZE,
    "axes.labelsize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
    "lines.linewidth": 2.5,
    "legend.frameon": False,
}
BAND_ALPHA = 0.2
# The legend sits above the figure, clear of the curves, titles and slice lines.
LEGEND = {
    "loc": "lower left",
    "bbox_to_anchor": (0, 1),
    "handlelength": 1,
    "handletextpad": 0.4,
}
LEGEND_MARKER = {"marker": "o", "linestyle": "none", "markersize": 14}
PANEL_SIZE = 8
BOX_ASPECT = 1
SEED_LINE = {"linewidth": 1, "alpha": 0.5}
WINDOW_LINE = {"color": "black", "linewidth": 2}
XLABEL = "Time Steps"
# Each component's legend label and color. Agent0 and DQN keep their colors from
# the parent experiments.
PANELS = {
    AGENT0_SEAQUEST: ("Agent0 in Seaquest", "tab:red"),
    DQN_ASTERIX: ("DQN in Minatar Asterix", "tab:blue"),
}
# Each component's y-axis range, shared by every panel so they compare directly.
Y_LIMITS = {
    AGENT0_SEAQUEST: (0, 10_000),
    DQN_ASTERIX: (0, 120),
}


def return_label(value):
    return f"{value / 1000:g}k" if value >= 1000 else str(value)


def step_label(step):
    return "0" if step == 0 else f"{step / 1_000_000:.10g}M"


def format_axes(ax, xticks, ylim):
    ax.set_xlabel(XLABEL)
    ax.set_xticks(xticks, [step_label(step) for step in xticks])
    # Long labels at the axis ends would run into the y-axis, so each grows inward.
    first, last = ax.get_xticklabels()
    first.set_horizontalalignment("left")
    last.set_horizontalalignment("right")
    ax.set_ylim(ylim)
    ax.set_yticks(ylim, [return_label(limit) for limit in ylim])
    ax.set_ylabel("Return", fontsize=YLABEL_SIZE)


def draw_band(ax, steps, interval, color):
    center, low, high = interval
    ax.fill_between(steps, low, high, color=color, alpha=BAND_ALPHA)
    ax.plot(steps, center, color=color)


def draw_seeds(ax, curve, color):
    steps, returns, _ = curve
    ax.plot(steps, returns.T, color=color, **SEED_LINE)


# Every panel of a component shares its y-range, so only the first column needs
# the y-axis label and ticks.
def hide_inner_y(axes):
    for ax in axes[:, 1:].flat:
        ax.set_ylabel("")
        ax.tick_params(axis="y", left=False, labelleft=False)


def add_legend(fig, label, color):
    handle = Line2D([], [], color=color, **LEGEND_MARKER)
    fig.legend([handle], [label], **LEGEND)


# Every seed's curve, then one summary of the seeds per row, over every span of
# time. Black lines mark the next column's span.
ROW_LABELS = [
    "Individual Runs",
    "Median, 95% TI",
    "Mean, STD",
    "Mean, STDERR",
    "Mean, 95% Bootstrap CI",
]


# The summaries of the seeds at every point, in row order.
def summaries(curve):
    _, returns, bootstrap = curve
    mean = returns.mean(axis=0)
    std = returns.std(axis=0, ddof=1)
    stderr = std / np.sqrt(len(returns))
    return [
        median_ti(returns),
        Interval(mean, mean - std, mean + std),
        Interval(mean, mean - stderr, mean + stderr),
        bootstrap,
    ]


def plot_summary_grid(name):
    label, color = PANELS[name]
    marked_spans = spans[name][1:] + [()]
    figsize = (PANEL_SIZE * len(spans[name]), PANEL_SIZE * len(ROW_LABELS))
    with plt.rc_context(STYLE):
        fig, axes = plt.subplots(len(ROW_LABELS), len(spans[name]), figsize=figsize)
        columns = zip(axes.T, spans[name], curves[name], marked_spans, strict=True)
        for column_axes, span, curve, marked in columns:
            seeds_ax, *band_axes = column_axes
            draw_seeds(seeds_ax, curve, color)
            for ax, interval in zip(band_axes, summaries(curve), strict=True):
                draw_band(ax, curve[0], interval, color)
            for ax in column_axes:
                ax.set_box_aspect(BOX_ASPECT)
                for step in marked:
                    ax.axvline(step, **WINDOW_LINE)
                format_axes(ax, list(span), Y_LIMITS[name])
        for ax, text in zip(axes[:, 0], ROW_LABELS, strict=True):
            ax.set_ylabel(text, fontsize=YLABEL_SIZE)

        hide_inner_y(axes)
        fig.tight_layout()
        add_legend(fig, label, color)
    return fig

In [ ]:
# Plot each summary of the seeds as a row over every span of time.
figures = {
    f"{name}_summary_grid": plot_summary_grid(name)
    for name in (AGENT0_SEAQUEST, DQN_ASTERIX)
}
plt.show()

In [ ]:
# Save every plotted figure as PDF and PNG in the plots directory.
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    for ext in ("pdf", "png"):
        fig.savefig(PLOTS_DIR / f"{name}.{ext}", bbox_inches="tight", dpi=200)
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")